# 🧠 Hybrid Classification Model with RoBERTa + Tabular Data
This tutorial shows how to combine `review_text` with tabular features like `rating`, `delivery_delay`, and `is_verified` to build a binary classifier using RoBERTa + feedforward layers.

In [ ]:
# 📦 Install necessary libraries
!pip install transformers datasets scikit-learn

In [ ]:
# ✅ Imports
import torch
from transformers import RobertaTokenizer, RobertaModel
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torch.nn as nn
import torch.optim as optim

In [ ]:
# 🧪 Simulated dataset mimicking Oracle table
texts = [
    "Great product, works as expected.",
    "Looks nice, haven't tried it yet.",
    "Very satisfied with the performance.",
    "Package arrived on time, not used yet.",
    "Loved it! Highly recommended.",
    "Nice packaging, still evaluating.",
    "Absolutely wonderful. Will buy again.",
    "Seems fine, not tested yet.",
    "Excellent quality and easy to use.",
    "Got it yesterday, looks okay so far."
]

labels = [
    "positive", "positive_irrelevant", "positive", "positive_irrelevant",
    "positive", "positive_irrelevant", "positive", "positive_irrelevant",
    "positive", "positive_irrelevant"
]

df = pd.DataFrame({
    'review_text': texts,
    'rating': np.random.randint(3, 6, size=10),
    'delivery_delay': np.random.randint(0, 5, size=10),
    'is_verified': np.random.choice([0, 1], size=10),
    'label': labels
})
df.head()

In [ ]:
# 🔤 Tokenize review_text
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
tokens = tokenizer(list(df['review_text']), padding=True, truncation=True, return_tensors='pt')

In [ ]:
# 🔎 RoBERTa embeddings (CLS token)
model = RobertaModel.from_pretrained("roberta-base")
with torch.no_grad():
    outputs = model(**tokens)
    cls_embeddings = outputs.last_hidden_state[:, 0, :].numpy()

In [ ]:
# 🧮 Normalize tabular features
scaler = StandardScaler()
tabular = scaler.fit_transform(df[['rating', 'delivery_delay', 'is_verified']])

In [ ]:
# 🔀 Combine text + tabular
X = np.concatenate([cls_embeddings, tabular], axis=1)
y = LabelEncoder().fit_transform(df['label'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
# 🤖 Simple classifier
class Classifier(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        return self.net(x)

model = Classifier(X.shape[1])
opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
# 🏋️ Train loop
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

for epoch in range(15):
    opt.zero_grad()
    pred = model(X_train_tensor)
    loss = loss_fn(pred, y_train_tensor)
    loss.backward()
    opt.step()
    print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")

In [ ]:
# 🔍 Inference on new review samples
def predict_review(text, rating, delivery_delay, is_verified):
    # Tokenize input text
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        roberta_output = model_roberta(**inputs)
        cls_embed = roberta_output.last_hidden_state[:, 0, :].numpy()

    # Prepare tabular input
    tabular_input = scaler.transform([[rating, delivery_delay, is_verified]])

    # Combine
    combined_input = np.concatenate([cls_embed, tabular_input], axis=1)
    combined_tensor = torch.tensor(combined_input, dtype=torch.float32)

    # Predict
    with torch.no_grad():
        logits = model(combined_tensor)
        probs = torch.softmax(logits, dim=1).numpy()[0]
        pred_class = np.argmax(probs)
        label = label_encoder.inverse_transform([pred_class])[0]
    return label, probs

# Example:
text = "Got it quickly, haven't used it yet."
label, confidence = predict_review(text, rating=5, delivery_delay=1, is_verified=1)
print(f"Prediction: {label} (Confidence: {confidence})")

In [ ]:
# 📊 Evaluate
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
with torch.no_grad():
    pred_logits = model(X_test_tensor)
    y_pred = torch.argmax(pred_logits, dim=1).numpy()

print(classification_report(y_test, y_pred, target_names=['positive', 'positive_irrelevant']))